In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mga input at output ng Estimator

<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekumenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Ang pahinang ito ay nagbibigay ng pangkalahatang-ideya ng mga input at output ng Qiskit Runtime Estimator primitive, na nagpapatupad ng mga workload sa mga compute resource ng IBM Quantum&reg;. Binibigyang-daan ka ng Estimator na mahusay na tumukoy ng mga vectorized na workload sa pamamagitan ng paggamit ng isang istraktura ng data na tinatawag na [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). Ginagamit ang mga ito bilang mga input sa method na [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) para sa Estimator primitive, na nagpapatupad ng tinukoy na workload bilang isang job. Pagkatapos, pagkatapos makumpleto ang job, ang mga resulta ay ibinabalik sa isang format na nakasalalay sa parehong mga PUB na ginamit pati na rin sa mga runtime option na tinukoy mula sa primitive.
## Mga input
Ang bawat PUB ay nasa format na ito:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

Ang opsyonal na `parameter values` ay maaaring isang listahan o isang solong parameter. Ang mga elemento mula sa mga observable at mga halaga ng parameter ay pinagsama sa pamamagitan ng pagsunod sa mga panuntunan ng NumPy broadcasting tulad ng inilarawan sa paksa ng [Mga input at output ng primitive](primitive-input-output#broadcasting-rules), at isang tinatantyang expectation value ang ibinabalik para sa bawat elemento ng broadcasted na hugis.

> **Note:** Kung ang input ay naglalaman ng mga sukat, ang mga ito ay binabalewala.

Para sa Estimator primitive, ang isang PUB ay maaaring maglaman ng hindi hihigit sa apat na halaga:
- Isang solong `QuantumCircuit`, na maaaring maglaman ng isa o higit pang mga object na [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Isang listahan ng isa o higit pang mga observable, na nagtatakda ng mga expectation value na tatantiyahin, na nakaayos sa isang array (halimbawa, isang solong observable na kinakatawan bilang isang 0-d array, isang listahan ng mga observable bilang isang 1-d array, at iba pa). Ang data ay maaaring nasa anumang isa sa mga format na `ObservablesArrayLike` tulad ng `Pauli`, `SparsePauliOp`, `PauliList`, o `str`.
   > **Note:** - Ang mga commuting observable **sa parehong PUB** ay pinagsama gamit ang [pamamaraang ito](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Ang mga commuting observable sa iba't ibang PUB, kahit na mayroon silang parehong circuit, ay hindi tinatantya gamit ang parehong sukat. Ang bawat PUB ay kumakatawan sa isang iba't ibang batayan para sa sukat, at samakatuwid, kinakailangan ang mga hiwalay na sukat para sa bawat PUB.
>    - Para tiyaking ang mga commuting observable ay tinatantya gamit ang parehong sukat, grupuhin ang mga ito sa loob ng parehong PUB.
- Isang koleksyon ng mga halaga ng parameter para i-bind ang circuit laban sa. Maaari itong tukuyin bilang isang solong array-like na object kung saan ang huling index ay sa mga object na `Parameter` ng circuit o tinanggal (o katumbas, itinakda sa `None`) kung ang circuit ay walang mga object na `Parameter`.
- (Opsyonal) Isang target na katumpakan para sa mga expectation value na tatantiyahin
---

Ang sumusunod na code ay nagpapakita ng isang halimbawang set ng mga vectorized na input sa primitive na `Estimator` at isinasagawa ang mga ito sa isang IBM&reg; backend bilang isang solong object na `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Mga output
Pagkatapos maipadala ang isa o higit pang PUB sa isang QPU para sa execution at matagumpay na makumpleto ang isang job, ang data ay ibinabalik bilang isang container object na [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) na na-access sa pamamagitan ng pagtawag sa method na `RuntimeJobV2.result()`.

Ang `PrimitiveResult` ay naglalaman ng isang iterable na listahan ng mga object na [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) na naglalaman ng mga resulta ng execution para sa bawat PUB.

Ang bawat elemento ng listahang ito ay tumutugma sa bawat PUB na isinumite sa method na `run()` ng primitive (halimbawa, ang isang job na isinumite na may 20 PUB ay magbabalik ng isang object na `PrimitiveResult` na naglalaman ng listahan ng 20 object na `PubResult`, isa na tumutugma sa bawat PUB).

Ang bawat `PubResult` para sa Estimator primitive ay naglalaman ng kahit isang array ng mga expectation value (`PubResult.data.evs`) at mga kaugnay na standard deviation (alinman sa `PubResult.data.stds` o `PubResult.data.ensemble_standard_error` depende sa `resilience_level` na ginamit), ngunit maaari itong maglaman ng higit pang data depende sa mga opsyon ng error mitigation na tinukoy.

Ang bawat object na `PubResult` ay nagtataglay ng parehong attribute na `data` at `metadata`.
    - Ang attribute na `data` ay isang customized na [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) na naglalaman ng aktwal na mga halaga ng sukat, mga standard deviation, at iba pa.
    - Ang `DataBin` ay may iba't ibang attribute depende sa hugis o istruktura ng kaugnay na PUB pati na rin sa mga opsyon ng error mitigation na tinukoy ng primitive na ginamit para isumite ang job (halimbawa, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) o [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Ang attribute na `metadata` ay naglalaman ng impormasyon tungkol sa mga opsyon ng runtime at error mitigation na ginamit (na ipinaliwanag sa ibang pagkakataon sa seksyon ng [Metadata ng resulta](#result-metadata) ng pahinang ito).

Ang sumusunod ay isang visual na balangkas ng istraktura ng data ng `PrimitiveResult` para sa output ng Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Sa simpleng salita, ang isang job ay nagbabalik ng isang object na `PrimitiveResult` at naglalaman ng listahan ng isa o higit pang object na `PubResult`. Ang mga object na `PubResult` na ito ay nag-iimbak ng data ng sukat para sa bawat PUB na isinumite sa job.

Ang sumusunod na snippet ng code ay naglalarawan ng format ng `PrimitiveResult` (at kaugnay na `PubResult`) para sa job na nilikha sa itaas.

In [ ]:
print(
    f"The result of the submitted job had {len(result)} "
    f"PUBs and has a value:\n {result}\n"
)
print(
    "The associated PubResult of this job has the following data bins:\n "
    "{result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets"
    "having shape (100, 2), where 2 is the number of parameters in the "
    "circuit, combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        "The expectation values measured from this PUB are: \n"
        "{result[0].data.evs}\n"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
